# **TAS: Tele Assistance System [DASA CASE STUDY 1]**

## **Summary**

This notebook is focused on three main objectives:
1. summarizing the key aspects of the Tele Assistance System (TAS) architecture and its adaptive capabilities in the context of telehealth services for chronic patients.
2. Modelling the TAS architecture using appropriate design notations and tools to visualize its components and interactions.
3. Simulating the TAS behavior under different scenarios to evaluate its performance and adaptability with Queue Network (QN) models.

The results will be used to evaluate the Dimensional Analysis for Software Architecture (DASA) methodology, its software tool (PyDASA) and its effectiveness in modelling and Quality Scenarios (QS) trade-off in self-adaptive-systems (SAS).

---

## **Software Architecture**
- TAS (Tele Assistance System) operates in a dynamic environment where service quality, availability, and user needs frequently change.
- The TAS is further subdivided into Controller and Target System subsystem components.
- The Controller is responsible for managing the overall system behavior, while the Target System focuses on executing specific tasks related to patient care.
- The TAS target systems follows a Service-oriented architecture (SOA) pattern.
- The TAS Controller follows a MAPE-K (Monitor-Analyze-Plan-Execute-Knowledge) feedback loop for self-adaptation.
- Adaptations focus on maintaining **reliability**, **performance**, and **compliance** with patient care standards (5 specific scenarios).
- ActivFORMS provides the runtime framework for model-based adaptation using runtime models, simulations, and verified decision-making.

---

_**NOTE: MORE DETAILS ON THE ARCHITECTURE IN THE ANALYTICAL MODELLING NOTEBOOK!.**_

---

## **Code**

_**SUMMARY:**_

This code is for the analytical solution of the Case Study (TAS) Dimensional Analysis Model and is structured as follows:
1. Analytical Dimensional Model (DA).
2. Importing necessary libraries and modules.
3. Loading DA default configuration.
4. Solving the DA analytically.
5. Solving the DA with Monte Carlo simulations.
6. Plotting the DA with the obtained metrics.
7. Loading DA 'optimal' configuration.
8. Solving the DA optimally.
9. Simulating the DA with the 'optimal' configuration.
10. Plotting the optimal DA with the obtained metrics.
11. Saving the results.
12. Comparing the analytical results (Default Vs. Optimal)
13. Visualizing the results.
14. Generating a summary report.

## **Queue Network Model**

<svg viewBox="0 0 4650 2000" width="1400" height="650">
    <!-- SVG content -->
    <image href="assets/cs1/img/04A - Queue Network.svg" alt="queue-net-diagram" />
    <div align="center"><em>Image 4. TAS Queue Network Diagram.</em></div>
</svg>

### **Necessary Imports**

In [1]:
# -*- coding: utf-8 -*-
# Native imports
import os
import re
import sys
import time
import copy
from typing import Union, List

# Third-party imports
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import matplotlib.pyplot as plt

# for plotting refined dimensionless chart
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import binned_statistic_2d
import matplotlib.ticker as ticker

# import queue network + models packages
from src.model.queueing import Queue
from src.model.analytical import solve_jackson_network, calculate_net_metrics

# import plot functions + grahics
from src.view.plots import plot_queue_network

# package import PyDASA
# config module
from pydasa.utils import config
# FDU modules
from pydasa.core.fundamental import Dimension
# FDU regex management
from pydasa.dimensional.framework import DimScheme
# Variable and Variable modules
from pydasa.core.parameter import Variable
# Dimensional Matrix Modelling module
from pydasa.dimensional.model import DimMatrix
# sensitivity analysis modules
from pydasa.analysis.scenario import DimSensitivity
from pydasa.handlers.influence import SensitivityHandler
# Monte Carlo Simulation modules
from pydasa.analysis.simulation import MonteCarloSim
from pydasa.handlers.practical import MonteCarloHandler

### **Function Definitions**

In [2]:
# Simple formatter for console output

def fmt(val: Union[int, float, np.number]) -> Union[str, np.ndarray]:
    """Format a number to 4 decimal places for console output.

    Args:
        val (Union[int, float, np.number, np.ndarray]): The value to format.

    Returns:
        Union[str, np.ndarray]: The formatted value as a string or an array of strings.
    """
    if isinstance(val, (int, float, np.number)):
        if np.isnan(val) or np.isinf(val):
            return str(val)
        return f"{val:.4f}"
    elif isinstance(val, np.ndarray):
        return np.array([fmt(x) for x in val])
    return val

In [3]:
# Load configuration from a CSV file
def load(path: str, fname: str) -> pd.DataFrame:
    """Load configuration from a CSV file.

    Args:
        path (str): The directory path where the CSV file is located.
        fname (str): The name of the CSV file to load.

    Returns:
        pd.DataFrame: A DataFrame containing the configuration data.
            CSV format:
                - node: <node_id>
                - miu: <mean_service_time>
                - c: <service_channels>
                - K: <buffer_capacity | max_queue_length>
                - lambda0: <initial_arrival_rate>
                - L0: <initial_queue_length>
                - pm: <matrix_routing_probabilities>
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Loading configuration from: {_file_path}")
    df = pd.read_csv(_file_path)
    return df

In [4]:
# save dataframes in CSV files
def save(path: str, fname: str, data: pd.DataFrame) -> None:
    """Save a DataFrame to a CSV file.

    Args:
        path (str): The directory path where the CSV file will be saved.
        fname (str): The name of the CSV file to save.
        data (pd.DataFrame): The DataFrame containing the data to save.
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Saving data to: {_file_path}")
    data.to_csv(_file_path, index=False)

In [5]:
# path = os.path.dirname(__file__)\
PATH = os.getcwd()
print(f"Notebook path: {PATH}")

Notebook path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies


In [6]:
# Folder names
asset_folder = "assets"
docs_folder = "docs"
img_folder = "img"
data_folder = "data"
report_folder = "reports"
results_folder = "results"
cs_folder = "cs1"

In [7]:
# setting case study data folder
file_path = os.path.join(PATH, data_folder, cs_folder)
print(f"Data path: {file_path}")

Data path: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1


### **Queue Model**
#### **Dimensional Analysis Solution**
##### **Base Configuration**

In [8]:
# Load configuration with mixed queue models
dflt_cs_cfg = load(file_path, "default_qn_model.csv")
print("Queue Network Configuration:")
dflt_cs_cfg.head()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1\default_qn_model.csv
Queue Network Configuration:


,node,name,type,miu,s,K,lambda0,L0,pm
0,1,TAS 1(1)*,M/M/1/K,900.0,1,1000,345.0,0,"[0.00,0.75,0.25,0.00,0.00,0.00,0.00,0.00,0.00,..."
1,2,TAS 2(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.33,0.33,0.33,0.00,0.00,0.00,..."
2,3,TAS 3(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.00,0.00,0.33,0.33,0.33,..."
3,4,MAS 1,M/M/1/K,180.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.12,0.00,0.00,0.00,0.00,0.00,..."
4,5,MAS 2,M/M/1/K,530.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.07,0.00,0.00,0.00,0.00,..."


In [9]:
# Load configuration with mixed queue models
dflt_da_cfg = load(file_path, "default_dim_variables.csv")
print("Dimension Variables Configuration:")
dflt_da_cfg.head()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1\default_dim_variables.csv
Dimension Variables Configuration:


,_idx,dm,_sym,_alias,_fwk,name,description,relevant,_cat,_dims,_units,_min,_max,_mean,_std_units,_std_min,_std_max,_std_mean,_dist_type,_dist_params
0,1,1,\lambda_{1},lambda_tas_1,SOFTWARE,TAS 1 arrival rate,Arrival rate for TAS 1,True,IN,E*T^-1,req/s,0,345.0,345.0,req/s,0,345.0,345.0,exponential,{'scale': 1/345}
1,2,1,\chi_{1},chi_tas_1,SOFTWARE,TAS 1 departure rate,Departure rate for TAS 1,True,CTRL,E*T^-1,req/s,0,345.0,345.0,req/s,0,345.0,345.0,exponential,{'scale': 1/345}
2,3,1,\mu_{1},mu_tas_1,SOFTWARE,TAS 1 service rate,Service rate for TAS 1,True,CTRL,E*T^-1,req/s,0,900.0,900.0,req/s,0,900.0,900.0,exponential,{'scale': 1/900}
3,4,1,c_{1},c_tas_1,SOFTWARE,TAS 1 resourses,Number of resources allocated in TAS 1,True,CTRL,E,req,0,1.0,1.0,req,0,1.0,1.0,uniform,"{'low': 0, 'high': 1}"
4,5,1,K_{1},K_tas_1,SOFTWARE,TAS 1 q-capacity,Allocated request capacity in TAS 1 memory buffer,True,IN,E,req,0,1000.0,1000.0,req,0,1000.0,1000.0,constant,{'constant':1000}


##### **FDU (Fundamental Design Unit)**

In [10]:
# setting up custom FDU for software architecture
tas_fwk = "SOFTWARE"
tas_scm = DimScheme(_sym="FDU_{TAS}",
                        _alias="fdu_tas",
                        _idx=0,
                        name="TAS FDUs",
                        _fwk=tas_fwk,
                        description="FDU schema for the TAS case study.")
# print(tas_scm)
print("=== TAS Fundamental Dimenional Unit (FDU) Schema ===")
for dim in tas_scm.fdus:
    print(f"\t'{dim.sym}': {dim.name}, [{dim.unit}]")

=== TAS Fundamental Dimenional Unit (FDU) Schema ===
	'T': Time, [s]
	'D': Data, [bit]
	'E': Effort, [req]
	'C': Connectivity, [node]
	'A': Capacity, [process]


In [11]:
tas_scm.update_global_config()
print("\n==== Checking Schema Regex ====")
print("\tWKNG_DFLT_FDU_PREC_LT:", config.WKNG_FDU_PREC_LT)
print("\tWKNG_FDU_RE:", config.WKNG_FDU_RE)
print("\tWKNG_POW_RE:", config.WKNG_POW_RE)
print("\tWKNG_NO_POW_RE:", config.WKNG_NO_POW_RE)
print("\tWKNG_FDU_SYM_RE:", config.WKNG_FDU_SYM_RE)


==== Checking Schema Regex ====
	WKNG_DFLT_FDU_PREC_LT: ['T', 'D', 'E', 'C', 'A']
	WKNG_FDU_RE: ^[TDECA](\^-?\d+)?(\*[TDECA](?:\^-?\d+)?)*$
	WKNG_POW_RE: \-?\d+
	WKNG_NO_POW_RE: [TDECA](?!\^)
	WKNG_FDU_SYM_RE: [TDECA]


#### **Dimensional Variables**

Based on the QN (Queue Network) model for the component + sequence diagrams


In [12]:
print("---- Dimensional Variables by Dimensional Matrix ----")
# create a dimensional set of variables from config file
# unique dimensional matrix
dimensional_matrix_list = dflt_da_cfg["dm"].unique().tolist()
# dictionary to hold dimensional variables
dimensional_vars_groups = {}

# group by dimensional matrix
for dm in dimensional_matrix_list:
    # filter by dimensional matrix number
    _vars = dflt_da_cfg[dflt_da_cfg["dm"] == dm]
    # filter by relevant attributes
    _vars = _vars[_vars["dm"] == dm]
    print(f"Dimensional Matrix: {dm}, with {_vars.shape[0]} relevant vars.")
    tdict = {}
    for var in _vars.to_dict(orient="records"):
        key = var["_sym"]
        # remove unnecessary keys
        var.pop("dm", None)
        # cast internal dict column
        if var.get("_dist_params") != None:
            data = eval(var.get("_dist_params"))
            dt = {"_dist_params": data}
            # print(data, type(data))
            var.update(dt)
        tdict[key] = Variable(**var)
    # add to dictionary
    dimensional_vars_groups[dm] = tdict

print(f"No. of Dimensional Variables Groups: {len(dimensional_vars_groups)}")

---- Dimensional Variables by Dimensional Matrix ----
Dimensional Matrix: 1, with 9 relevant vars.
Dimensional Matrix: 2, with 9 relevant vars.
Dimensional Matrix: 3, with 9 relevant vars.
Dimensional Matrix: 4, with 9 relevant vars.
Dimensional Matrix: 5, with 9 relevant vars.
Dimensional Matrix: 6, with 9 relevant vars.
Dimensional Matrix: 7, with 9 relevant vars.
Dimensional Matrix: 8, with 9 relevant vars.
Dimensional Matrix: 9, with 9 relevant vars.
Dimensional Matrix: 10, with 9 relevant vars.
Dimensional Matrix: 11, with 9 relevant vars.
Dimensional Matrix: 12, with 9 relevant vars.
Dimensional Matrix: 13, with 9 relevant vars.
No. of Dimensional Variables Groups: 13


In [13]:
print("---- Configure Simulation Distribution Function for Variables ----")
# im tired so lambda functions are in order
for dm in dimensional_vars_groups:
    for key, var in dimensional_vars_groups[dm].items():
        # print(var)
        # if the distriburion is constant
        if var._dist_type == "constant":
            _constant = var._dist_params.get("constant")
            var._dist_func = lambda: _constant
        # with uniform distribution
        if var._dist_type == "uniform":
            _low = var._dist_params.get("low")
            _high = var._dist_params.get("high")
            var._dist_func = lambda: np.random.uniform(_low, _high)
        # with exponential distribution
        if var._dist_type == "exponential":
            _scale = var._dist_params.get("scale")
            var._dist_func = lambda:  np.random.exponential(_scale)


---- Configure Simulation Distribution Function for Variables ----


In [14]:
# Load configuration with mixed queue models
dflt_cs_cfg = load(file_path, "default_qn_model.csv")
print("Queue Network Configuration:")
# print(dflt_cs_cfg)
dflt_cs_cfg.head()

Loading configuration from: c:\Users\Felipe\OneDrive\Documents\GitHub\DASA-Design\PyDASA-Case-Studies\data\cs1\default_qn_model.csv
Queue Network Configuration:


,node,name,type,miu,s,K,lambda0,L0,pm
0,1,TAS 1(1)*,M/M/1/K,900.0,1,1000,345.0,0,"[0.00,0.75,0.25,0.00,0.00,0.00,0.00,0.00,0.00,..."
1,2,TAS 2(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.33,0.33,0.33,0.00,0.00,0.00,..."
2,3,TAS 3(1)*,M/M/1/K,700.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.00,0.00,0.33,0.33,0.33,..."
3,4,MAS 1,M/M/1/K,180.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.12,0.00,0.00,0.00,0.00,0.00,..."
4,5,MAS 2,M/M/1/K,530.0,1,1000,0.0,0,"[0.00,0.00,0.00,0.00,0.07,0.00,0.00,0.00,0.00,..."


In [15]:
# extract parameters from the configuration DataFrame
# and casting them to proper types
nodes = list(dflt_cs_cfg["node"].values.astype(int))
names = list(dflt_cs_cfg["name"].values)
types = list(dflt_cs_cfg["type"].values)
lambda_zs = list(dflt_cs_cfg["lambda0"].values)

# Convert string representations of arrays to actual numpy arrays
# and create routing matrix P
prob = []
for pm_str in dflt_cs_cfg["pm"].values:
    pm_values = pm_str.strip("[]").split(",")
    pm_values = [float(val) for val in pm_values]
    prob.append(pm_values)
P = np.array((prob))


#### **Dimensional Model**

In [16]:
print("---- Creating Dimensional Model (Matrix) ----")
dimensional_model_groups = {}

# group by dimensional matrix
for dm in dimensional_matrix_list:
    tas_vars = dimensional_vars_groups[dm]
    print(f"Dimensional Matrix ID: {dm}, N Variables: {len(tas_vars)}")
    dimensional_model_groups[dm] = DimMatrix(_fwk="SOFTWARE",
                                    _idx=dm,
                                    _framework=tas_scm)
    # setting Dimensional Model variables
    dimensional_model_groups[dm].variables = tas_vars
    _msg = "\tSetting parameters for the DA, "
    _msg += f"N Variables: {len(dimensional_model_groups[dm].variables)}. "
    # _msg += f"Variables: {dimensional_model_groups[dm].variables.keys()}"
    print(_msg)

    # setting Dimensional Model relevance list
    dimensional_model_groups[dm].relevant_lt = tas_vars
    _msg = "\tSetting the relevance list for the DA, "
    _msg += f"N Variables: {len(dimensional_model_groups[dm].relevant_lt)}. "
    # _msg += f"Variables: {dimensional_model_groups[dm].relevant_lt.keys()}"
    print(_msg)


---- Creating Dimensional Model (Matrix) ----
Dimensional Matrix ID: 1, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 2, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 3, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 4, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 5, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 6, N Variables: 9
	Setting parameters for the DA, N Variables: 9. 
	Setting the relevance list for the DA, N Variables: 7. 
Dimensional Matrix ID: 7, N Variables: 9
	Setting parameters for the DA,

In [17]:
print("--- Solving Dimensional Model (Matrix) ----")
# solving each dimensional model
for dm in dimensional_model_groups:
    print(f"Solving Dimensional Model ID: {dm}")
    # Here you would implement the logic to solve the dimensional model
    print(f"\tCreating Matrix for Dimensional Model ID: {dm}")
    dimensional_model_groups[dm].create_matrix()
    dimensional_model_groups[dm].solve_matrix()
    n = len(dimensional_model_groups[dm].coefficients)
    print(f"\tDimensional Model ID: {dm}, N Coefficients: {n}")
    print(f"\tFinished Solving Dimensional Model ID: {dm}")
    # print(len(dimensional_model_groups[dm].coefficients), "\n")
    # print(dimensional_model_groups[dm].coefficients, "\n")
    # print(dimensional_model_groups, "\n")

--- Solving Dimensional Model (Matrix) ----
Solving Dimensional Model ID: 1
	Creating Matrix for Dimensional Model ID: 1
	Dimensional Model ID: 1, N Coefficients: 5
	Finished Solving Dimensional Model ID: 1
Solving Dimensional Model ID: 2
	Creating Matrix for Dimensional Model ID: 2
	Dimensional Model ID: 2, N Coefficients: 5
	Finished Solving Dimensional Model ID: 2
Solving Dimensional Model ID: 3
	Creating Matrix for Dimensional Model ID: 3
	Dimensional Model ID: 3, N Coefficients: 5
	Finished Solving Dimensional Model ID: 3
Solving Dimensional Model ID: 4
	Creating Matrix for Dimensional Model ID: 4
	Dimensional Model ID: 4, N Coefficients: 5
	Finished Solving Dimensional Model ID: 4
Solving Dimensional Model ID: 5
	Creating Matrix for Dimensional Model ID: 5
	Dimensional Model ID: 5, N Coefficients: 5
	Finished Solving Dimensional Model ID: 5
Solving Dimensional Model ID: 6
	Creating Matrix for Dimensional Model ID: 6
	Dimensional Model ID: 6, N Coefficients: 5
	Finished Solving Di

#### **Dimensionless Coefficients**

In [18]:

print("---- Indexing Coefficients and Sensitivity Groups ----")
# Coefficient Groups
coefficient_groups = {}
# Sensitivity Groups
sensitivity_groups = {}

for dm in dimensional_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in dimensional_model_groups[dm].coefficients.items():
    # creating sensitivity group
    sensitivity_groups[dm] = SensitivityHandler(
        _idx=dm,
        _sym=f"$SA_{{TAS_{dm}}}$",
        name=f"Sensitivity Analysis for TAS DM {dm}",
        description=f"Sensitivity Analysis TAS Dimensional Model ID {dm}.",
        _variables=dimensional_vars_groups[dm],
        _coefficients=dimensional_model_groups[dm].coefficients,
    )
    # indexing coefficient groups
    keys = dimensional_model_groups[dm].coefficients.keys()
    values = dimensional_model_groups[dm].coefficients.values()
    coefficient_groups[dm] = dict(zip(keys, values))
    n = len(coefficient_groups[dm])
    print(f"\tIndexed {n} Coefficients for DM ID: {dm}\n")


---- Indexing Coefficients and Sensitivity Groups ----
Indexing Coefficient Groups from the Dimensional Model ID: 1
	Indexed 5 Coefficients for DM ID: 1

Indexing Coefficient Groups from the Dimensional Model ID: 2
	Indexed 5 Coefficients for DM ID: 2

Indexing Coefficient Groups from the Dimensional Model ID: 3
	Indexed 5 Coefficients for DM ID: 3

Indexing Coefficient Groups from the Dimensional Model ID: 4
	Indexed 5 Coefficients for DM ID: 4

Indexing Coefficient Groups from the Dimensional Model ID: 5
	Indexed 5 Coefficients for DM ID: 5

Indexing Coefficient Groups from the Dimensional Model ID: 6
	Indexed 5 Coefficients for DM ID: 6

Indexing Coefficient Groups from the Dimensional Model ID: 7
	Indexed 5 Coefficients for DM ID: 7

Indexing Coefficient Groups from the Dimensional Model ID: 8
	Indexed 5 Coefficients for DM ID: 8

Indexing Coefficient Groups from the Dimensional Model ID: 9
	Indexed 5 Coefficients for DM ID: 9

Indexing Coefficient Groups from the Dimensional Model

#### **Sensitivity Analysis**

In [19]:
print("---- Executing Sensitivity Analysis ----")

for dm in sensitivity_groups:
    print(f"Executing Sensitivity Analysis for: {dm}")
    print("\tExecuting Symbolic Analysis...")
    sensitivity_groups[dm].analyze_symbolic(val_type="mean")
    print("\tExecuting Numerical Analysis...")
    sensitivity_groups[dm].analyze_numeric(n_samples=10000)
    print(f"Finishing Analysis for: {dm}\n")

---- Executing Sensitivity Analysis ----
Executing Sensitivity Analysis for: 1
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 1

Executing Sensitivity Analysis for: 2
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 2

Executing Sensitivity Analysis for: 3
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 3

Executing Sensitivity Analysis for: 4
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 4

Executing Sensitivity Analysis for: 5
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 5

Executing Sensitivity Analysis for: 6
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 6

Executing Sensitivity Analysis for: 7
	Executing Symbolic Analysis...
	Executing Numerical Analysis...
Finishing Analysis for: 7

Executing Sensitivity Analysis for: 8
	Executing 

In [20]:
print("---- Printing Coefficient Sensitivity Analysis ----")
for dm in sensitivity_groups:
    print(f"Sensitivity Analysis Results for Dimensional Model ID: {dm}")
    for key, val in sensitivity_groups[dm].results.items():
        print(f"\t{key}: {val}")

---- Printing Coefficient Sensitivity Analysis ----
Sensitivity Analysis Results for Dimensional Model ID: 1
	SEN_{\Pi_{0}}: {'S1': [np.float64(0.4274052535664964), np.float64(0.4277082928802679)], 'ST': [np.float64(0.5716100531914725), np.float64(0.5713063216225107)], 'S1_conf': [np.float64(0.0015641719179799796), np.float64(0.0015636418684033496)], 'ST_conf': [np.float64(0.014163031340222064), np.float64(0.011976293817295603)], 'names': ['W_{1}', '\\lambda_{1}']}
	SEN_{\Pi_{1}}: {'S1': [np.float64(0.000893383986913393), np.float64(0.020213916339009038)], 'ST': [np.float64(0.4759384213029083), np.float64(0.8770345518268582)], 'S1_conf': [np.float64(0.0012759763256891014), np.float64(0.0012462427376105304)], 'ST_conf': [np.float64(0.01351495547788132), np.float64(0.008929165813400215)], 'names': ['\\chi_{1}', '\\lambda_{1}']}
	SEN_{\Pi_{2}}: {'S1': [np.float64(0.005535408358920736), np.float64(0.001492738321470422)], 'ST': [np.float64(0.8755902768111147), np.float64(0.38062537984703426

#### **Monte Carlo Simulation**

In [21]:
print("---- Create Monte Carlo Simulations ----")
monte_carlo_groups = {}

for dm in dimensional_model_groups:
    print(f"Indexing Coefficient Groups from the Dimensional Model ID: {dm}")
    # for pi, coef in dimensional_model_groups[dm].coefficients.items():
    # creating sensitivity group
    monte_carlo_groups[dm] = MonteCarloHandler(
        _idx=dm,
        _fwk="SOFTWARE",
        _sym=f"$MC_{{TAS_{dm}}}$",
        name=f"Monte Carlo Simulation for TAS DM {dm}",
        description=f"Monte Carlo Simulation TAS Dimensional Model ID {dm}.",
        _variables=dimensional_model_groups[dm]._relevant_lt,
        _coefficients=dimensional_model_groups[dm].coefficients,
    )
    n = len(monte_carlo_groups[dm].coefficients)
    print(f"\tIndexed {n} Coefficients Monte Carlo: {dm}\n")

---- Create Monte Carlo Simulations ----
Indexing Coefficient Groups from the Dimensional Model ID: 1
	Indexed 5 Coefficients Monte Carlo: 1

Indexing Coefficient Groups from the Dimensional Model ID: 2
	Indexed 5 Coefficients Monte Carlo: 2

Indexing Coefficient Groups from the Dimensional Model ID: 3
	Indexed 5 Coefficients Monte Carlo: 3

Indexing Coefficient Groups from the Dimensional Model ID: 4
	Indexed 5 Coefficients Monte Carlo: 4

Indexing Coefficient Groups from the Dimensional Model ID: 5
	Indexed 5 Coefficients Monte Carlo: 5

Indexing Coefficient Groups from the Dimensional Model ID: 6
	Indexed 5 Coefficients Monte Carlo: 6

Indexing Coefficient Groups from the Dimensional Model ID: 7
	Indexed 5 Coefficients Monte Carlo: 7

Indexing Coefficient Groups from the Dimensional Model ID: 8
	Indexed 5 Coefficients Monte Carlo: 8

Indexing Coefficient Groups from the Dimensional Model ID: 9
	Indexed 5 Coefficients Monte Carlo: 9

Indexing Coefficient Groups from the Dimensional M

In [22]:
print("---- Execute Monte Carlo Simulations ----")
for dm in monte_carlo_groups:
    print(f"\tStarting Montecarlo Simulation group: {dm}")
    monte_carlo_groups[dm]._create_distributions()
    monte_carlo_groups[dm]._create_simulations()
    monte_carlo_groups[dm].simulate(n_samples=1000)
    print(f"\tFinishing simulation for group: {dm}\n")

---- Execute Monte Carlo Simulations ----
	Starting Montecarlo Simulation group: 1
	Finishing simulation for group: 1

	Starting Montecarlo Simulation group: 2
	Finishing simulation for group: 2

	Starting Montecarlo Simulation group: 3
	Finishing simulation for group: 3

	Starting Montecarlo Simulation group: 4
	Finishing simulation for group: 4

	Starting Montecarlo Simulation group: 5
	Finishing simulation for group: 5

	Starting Montecarlo Simulation group: 6
	Finishing simulation for group: 6

	Starting Montecarlo Simulation group: 7
	Finishing simulation for group: 7

	Starting Montecarlo Simulation group: 8
	Finishing simulation for group: 8

	Starting Montecarlo Simulation group: 9
	Finishing simulation for group: 9

	Starting Montecarlo Simulation group: 10
	Finishing simulation for group: 10

	Starting Montecarlo Simulation group: 11
	Finishing simulation for group: 11

	Starting Montecarlo Simulation group: 12
	Finishing simulation for group: 12

	Starting Montecarlo Simulat

In [23]:
print("---- Monte Carlo Simulation Report ----")
for md in monte_carlo_groups:
    mcg = monte_carlo_groups[md]
    print(f"Monte Carlo Group: {md}, with {mcg._iterations} samples.")
    for pi, results in mcg._results.items():
        for k, v in results.items():
            if k == "statistics":
                coef = mcg._coefficients[pi].pi_expr
                print(f"\tCoefficient: {pi} = {coef}")
                # print(f"Result for {k}: {v}")
                stats = v
                # print(f"\tStats: {stats}\n")
                # for a, b in stats.items():
                #     print(f"\t\t{a}: {b}")
            # else:
            #     print(f"Other result for {k}: {v.shape}")

---- Monte Carlo Simulation Report ----
Monte Carlo Group: 1, with 1000 samples.
	Coefficient: \Pi_{0} = \lambda_{1}*W_{1}
	Coefficient: \Pi_{1} = \frac{\chi_{1}}{\lambda_{1}}
	Coefficient: \Pi_{2} = \frac{\mu_{1}}{\lambda_{1}}
	Coefficient: \Pi_{3} = \frac{c_{1}}{K_{1}}
	Coefficient: \Pi_{4} = \frac{L_{1}}{K_{1}}
Monte Carlo Group: 2, with 1000 samples.
	Coefficient: \Pi_{0} = \lambda_{2}*W_{2}
	Coefficient: \Pi_{1} = \frac{\chi_{2}}{\lambda_{2}}
	Coefficient: \Pi_{2} = \frac{\mu_{2}}{\lambda_{2}}
	Coefficient: \Pi_{3} = \frac{c_{2}}{K_{2}}
	Coefficient: \Pi_{4} = \frac{L_{2}}{K_{2}}
Monte Carlo Group: 3, with 1000 samples.
	Coefficient: \Pi_{0} = \lambda_{3}*W_{3}
	Coefficient: \Pi_{1} = \frac{\chi_{3}}{\lambda_{3}}
	Coefficient: \Pi_{2} = \frac{\mu_{3}}{\lambda_{3}}
	Coefficient: \Pi_{3} = \frac{c_{3}}{K_{3}}
	Coefficient: \Pi_{4} = \frac{L_{3}}{K_{3}}
Monte Carlo Group: 4, with 1000 samples.
	Coefficient: \Pi_{0} = \lambda_{4}*W_{4}
	Coefficient: \Pi_{1} = \frac{\chi_{4}}{\lambda_{

#### **Data Analysis**

The steps are:

1) Extract and organize simulation data into a unique DataFrame.
2) Add metadata and basic statistics.
3) Save the dataframe to a CSV file.
4) Create a basic dimensionless plot (similar to Moody's chart)

##### **Data Post-Processing**


In [ ]:
print("---- Monte Carlo Simulation Post-Processing ----")
# detailed report
# coefficient global index
i = 0

# data for unified dataframes
exp_records = {}

# simulation statistical data
stats_records = []

# global coefficient name = coefficient formula
pi_coef = {}

# iterate over monte carlo groups
for mcg in monte_carlo_groups:
    # monte carlo group header
    n = monte_carlo_groups[mcg]._iterations
    _msg = f"\tMonte Carlo Simulation Group: {mcg}, with {n} samples."
    print(_msg)
    _vars = monte_carlo_groups[mcg].variables.keys()
    # _vars = ", ".join(v for v in _vars)
    _msg = f"\tN Variables: {len(_vars)}, Var := {list(_vars)}"
    print(_msg)

    _coefs = monte_carlo_groups[mcg].coefficients
    print(f"\tNo. of Pi-Coefficients in the group: {len(_coefs)}")
    # iterating the Pi names of the group coefficients
    for j, pi in enumerate(_coefs):
        # getting the coefficient data
        coef = _coefs[pi]
        # coefficient header
        # getting the coefficient relevant dimensional variables
        cvars = list(coef.var_dims.keys())
        # print(f"\t\tCoefficient: {coef.name}: {pi} = {coef.pi_expr}")
        # print(f"\t\t\tGlobal Idx: {i}, Inputs := {cvars}, Size: {len(cvars)}")

        # rename Pi for DataFrame column with global idx
        pi_rename = str(pi)
        pi_rename = pi_rename.replace(str(j), str(i))
        # print(f"\t\t\tRenaming Coef. FROM: {pi} TO: {pi_rename}")
        pi_coef[pi_rename] = coef.pi_expr

        # get simulation results
        simulations = monte_carlo_groups[mcg].get_simulation(pi)
        # print(type(simulations))
        if simulations:
            # exporting results
            results = simulations.extract_results()
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            for key in results:
                new_key = key.replace(pi, pi_rename)
                # dictionary[new_key] = dictionary.pop(old_key)
                results[new_key] = results.pop(key)
            # print(f"\t\t\tResults keys: {list(results.keys())}")
            
            exp_records.update(results)

            # exporting statistics
            stats = {
                "name": pi_rename,
                "coef": coef.pi_expr
            }
            # print(f"\t\t\tStatistics before update: {stats}")
            stats.update(simulations.statistics)
            # print(f"\t\t\tStatistics after update: {stats}")
            stats_records.append(stats)

        # incrementing global index
        i += 1
        # print(i)
    print(f"\tEnding Monte Carlo group No.: {mcg}...\n")

# creating DataFrame from records
exp_df = pd.DataFrame(exp_records)
stats_df = pd.DataFrame(stats_records)

In [ ]:
print("---- Monte Carlo Simulation DataFrames ----")
print(exp_df.shape)
exp_df.info()
exp_df.head()

In [ ]:
print("---- Statistics DataFrame ----")
print(stats_df.shape)
stats_df.info()
stats_df.head()

In [ ]:
print("---- Pi-Coefficients DataFrame ----")
# Extracting just the coefficients from the experimental DataFrame
pi_keys = list(pi_coef.keys())
print(f"Total No. of Pi-Coefficients: {len(pi_keys)}")
pi_df = pd.DataFrame(exp_df[pi_keys])
pi_cols = {}
# renaming columns with coef = formula
for k, v in pi_coef.items():
    pi_cols[k] = f"{k}={v}"
# renaming columns
pi_df.rename(columns=pi_cols, inplace=True)
# checking the Pi DataFrame
print(pi_df.shape)
pi_df.info()
pi_df.head()

In [ ]:
# make generic columns for dimensionless coefficients
sys_name = "TAS"
sys_cols = [re.sub(r"\d+", sys_name, col) for col in pi_coef.values()]
# removing redundant entries, keeping order
sys_cols = list(dict.fromkeys(sys_cols))
for i in range(len(sys_cols)):
    formula = sys_cols[i]
    sys_cols[i] = f"\\Pi_{{{i+1}}}={formula}"
print(f"System generic columns: {len(sys_cols)}, {sys_cols}")

In [ ]:
print("---- System DataFrame ----")
print(f"Pi Data shape: {pi_df.shape}")
# dictionary for system records
sys_records = {}

# iterate over the generic columns
for i, col in enumerate(sys_cols):
    print(f"\tSetting column: {col} in idx {i}")
    # mod = i % len(sys_cols)
    # print(f"\tidx: {col}, Sys Col: {col}, Module: {mod}")
    # iterate over the pi dataframe columns
    for j in range(i, pi_df.shape[1], len(sys_cols)):
        c = pi_df.columns[j]
        data = pi_df[c].values
        # print(f"\t\textracting data for column: {c}, size: {len(data)}")
        if col not in sys_records:
            print(f"\t\tCreating new column: {col}!!!...")
            sys_records[col] = data
            # break
        elif col in sys_records:
            print(f"\t\tAppending data to existing column: {col}")
            sys_records[col] = np.concatenate((sys_records[col], data))
            # break

In [ ]:
print("---- Checking System Records Info ----")
for k, v in sys_records.items():
    print(f"{k}: {len(v)}")
    
print("---- System DataFrame ----")
sys_df = pd.DataFrame(sys_records)
print(sys_df.shape)
sys_df.info()
sys_df.head()

In [ ]:
# creating derived Pi-Coefficients
print("---- Creating Derived Pi-Coefficients ----")
# derived coefficients columns

# working variables for derived coefficients
print("Creating Base Coefficients: \\Pi_{{1}}, \\Pi_{{2}}, \\Pi_{{3}}")

# Pi_1 = lambda * W
temp = f"\\Pi_{{1}}=\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
pi_1 = sys_df[temp]

# Pi_2 = chi / lambda
temp = f"\\Pi_{{2}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_2 = sys_df[temp]

# Pi_3 = mu / lambda
temp = f"\\Pi_{{3}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_3 = sys_df[temp]

# 1 --->> System/Component Congestion Coef: Pi_6 = Pi_2 * Pi_3^{-1} * Pi_1
print("Creating Congestion Coefficient: \\Pi_{{6}}")
numerator = f"\\chi_{{{sys_name}}}*\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
denominator = f"\\mu_{{{sys_name}}}"

new_coef = f"\\Pi_{{6}}=\\frac{{{numerator}}}{{{denominator}}}"
# Pi_6 = chi * Lambda * W / mu
sys_df[new_coef] = pi_2 * (pi_3**-1) * pi_1

# working variables for derived coefficients
print("Creating Base Coefficients: \\Pi_{{4}}, \\Pi_{{5}}")
# Pi_4 = c / K
temp = f"\\Pi_{{4}}=\\frac{{c_{{{sys_name}}}}}{{K_{{{sys_name}}}}}"
pi_4 = sys_df[temp]

# Pi_5 = L / K
temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{K_{{{sys_name}}}}}"
pi_5 = sys_df[temp]

# 2 --->> Queue Occupancy Coef: Pi_7 = Pi_5 * Pi_4^{-1}
print("Creating Queue Occupancy Coefficient: \\Pi_{{7}}")
numerator = f"L_{{{sys_name}}}"
denominator = f"C_{{{sys_name}}}"

new_coef = f"\\Pi_{{7}}=\\frac{{{numerator}}}{{{denominator}}}"
# Pi_7 = L / C
sys_df[new_coef] = pi_5 * (pi_4**-1)

In [ ]:
sys_df.info()
sys_df

In [ ]:
# get statistics from one coefficient
rhos_range = [
    0.05,   # 0
    0.1,    # 1
    0.2,    # 2
    0.3,    # 3
    0.4,    # 4
    0.5,    # 5
    0.6,    # 6
    0.7,    # 7
    0.8,    # 8
    0.9,    # 9
    0.95,   # 10
    1.0,    # 11
    1.2,    # 12
    1.5,    # 13
    2.0,    # 14
    3.0,    # 15
]
print(len(rhos_range))

In [ ]:
# Create directories if they dont exist
os.makedirs("data/CS-1-HealthTAS/img", exist_ok=True)

# Figure setup with appropriate size, resolution and white background
fig, ax = plt.subplots(figsize=(12, 9), dpi=300, facecolor="white")
ax.set_facecolor("white")

# Get first two coefficients for the plot axes
pi_list = sys_df.columns.tolist()
pi_x = pi_list[6]       # occupancy coefficient for x-axis
pi_y = pi_list[5]       # congestion coefficient for y-axis
pi_z = pi_list[4]      # available buffer coefficient for z-axis

# Calculate utilization ratio (common in queueing theory)
sys_df["$\\rho$"] = 1 / sys_df[pi_list[2]]

# Group data by utilization ratio to create characteristic curves
utilization_ranges = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.0,
    1.2,
    1.5,
    2.0
]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(utilization_ranges)))
# alts: viridis_r, plasma, inferno, magma, cividis, coolwarm, Spectral

# Add hexbin background to show data density (like Moody chart)
hb = plt.hexbin(
    sys_df[pi_x],
    sys_df[pi_y],
    gridsize=40,
    cmap="Greens",
    alpha=0.1,
    mincnt=1,
    reduce_C_function=np.mean
)

# Plot characteristic curves for different utilization ranges
for i, util in enumerate(utilization_ranges):
    # Filter data near this utilization value
    mask = (sys_df["$\\rho$"] >= util - 0.05) & \
           (sys_df["$\\rho$"] <= util + 0.05)
    subset = sys_df[mask]

    if len(subset) > 50:  # Only draw if we have enough points
        # Sort by x-value for smooth curves
        subset = subset.sort_values(by=pi_x)

        # Create a polynomial fit (3rd degree works well for most curves)
        if len(subset) > 10:
            z = np.polyfit(np.log10(subset[pi_x]), np.log10(subset[pi_y]), 3)
            p = np.poly1d(z)

            # Create smooth x values for the line
            x_smooth = np.logspace(
                np.log10(subset[pi_x].min()),
                np.log10(subset[pi_x].max()),
                100
            )

            # Calculate predicted y values (convert back from log space)
            y_smooth = 10**p(np.log10(x_smooth))

            # Plot the trend line
            plt.plot(
                x_smooth,
                y_smooth,
                "-",
                linewidth=1.25,
                alpha=1.0,
                color=colors[i],
                label=f"ρ = {util:.2f}"
            )

# # Add contour lines for queue length
# # make a dicision in z-axis of 5 log levels
# # queue_space_vals = np.logspace(-3, 3, 7)
# queue_space_vals = np.logspace(-7,
#                                7,
#                                num=100000,
#                                endpoint=True,
#                                base=10.0,)
# # [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]
# # [10, 20, 30, 40, 50]
# # [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]
# contour = plt.tricontour(
#     sys_df[pi_x],
#     sys_df[pi_y],
#     sys_df[pi_z],

#     levels=queue_space_vals,
#     colors="black",
#     linestyles="dashed",
#     linewidths=0.5,
#     alpha=0.70
# )

# # Add contour labels in black
# plt.clabel(contour, inline=True, fontsize=10, fmt="n=%4.0f", colors="black")

# Set up log scales (standard for Moody-like charts)
plt.xscale("log")
plt.yscale("log")

# Adjust these values as needed to get the desired range
y_min = sys_df[pi_y].quantile(0.05)  # Use 5th percentile of data as minimum
y_max = sys_df[pi_y].quantile(0.95)  # Use 95th percentile of data as maximum
plt.ylim(y_min, y_max)

# Add grid with minor lines - lighter color for better visibility on white background
plt.grid(True, which="both", ls="-", color="lightgray", alpha=0.75)
plt.grid(True, which="minor", ls=":", color="lightgray", alpha=0.75)
plt.minorticks_on()

# Format tick labels for better readability
formatter = ticker.LogFormatterMathtext(base=10)
plt.gca().xaxis.set_major_formatter(formatter)
plt.gca().yaxis.set_major_formatter(formatter)

# Make sure ticks and labels are black
ax.tick_params(axis="both", colors="black")

# Add descriptive labels and title in black
plt.xlabel(
    f"System Congestion: ${pi_x}$", fontsize=14, color="black")
plt.ylabel(
    f"Queue Occupancy: ${pi_y}$", fontsize=14, color="black")
plt.title("System Performance Chart", fontsize=16, color="black")

# Add legend with clear styling
legend = plt.legend(
    title="System Utilization ($\\rho$)",
    loc="best",
    fontsize=12,
    framealpha=0.9
)
legend.get_title().set_color("black")

# Bottom-Left (Low Y, Low X)
plt.text(
    sys_df[pi_x].quantile(0.05),  # X position at 5th percentile
    sys_df[pi_y].quantile(0.10),  # Y position at 10th percentile
    "Low Congestion\nLow Occupancy",
    fontsize=11,
    ha="left",
    va="bottom",
    color="black",
    bbox=dict(facecolor="white", alpha=0.8, boxstyle="round", edgecolor="gray")
)

# Bottom-Right (Low Y, High X)
plt.text(
    sys_df[pi_x].quantile(0.95),  # X position at 95th percentile
    sys_df[pi_y].quantile(0.90),  # Y position at 90th percentile
    "High Congestion\nHigh Occupancy",
    fontsize=11,
    ha="right",
    va="bottom",
    color="black",
    bbox=dict(facecolor="white", alpha=0.8, boxstyle="round", edgecolor="gray")
)

# Save and display with white background
plt.tight_layout()
plt.savefig("data/CS-1-HealthTAS/img/tas_behaviour_chart_2222.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Solve the network analytically
# first node metrics
dflt_analyt_nd_metrics = solve_jackson_network(mius, lambda_zs, queues, P)

# then network metrics
dflt_analyt_net_metrics = calculate_net_metrics(dflt_analyt_nd_metrics)

In [ ]:
print("\n--- Analytical Jackson Network (Node Metrics) ---")
# print(dflt_analyt_nd_metrics)
dflt_analyt_nd_metrics.head()

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_analytical_node_metrics.csv", dflt_analyt_nd_metrics)

In [ ]:
print("\n--- Analytical Jackson Network (Network-wide Metrics) ---")
# print(dflt_analyt_net_metrics)
dflt_analyt_net_metrics.head()

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_analytical_net_metrics.csv", dflt_analyt_net_metrics)

In [ ]:
# plotting the queue network with metrics on each node
# data table column names
col_names =[
    "Component",
    "$\mathbf{\\lambda}$ [req/s]",
    "$\mathbf{\\mu}$ [req/s]",
    "$\mathbf{\\rho}$",
    "$\mathbf{L}$ [req]",
    "$\mathbf{L_q}$ [req]",
    "$\mathbf{W}$ [s/req]",
    "$\mathbf{W_q}$ [s/req]"
]

node_names = dflt_cs_cfg["name"].values.tolist()
print(f"Datatable column names: {col_names}")
print(f"Node names: {node_names}")  

# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# Plot the queue network
plot_queue_network(P,
                   dflt_analyt_net_metrics,
                   dflt_analyt_nd_metrics,
                   node_names,
                   col_names,
                   file_path,
                   "dflt_analytical_qn_diagram.png")

##### **Optimized Configuration**

In [ ]:
# Load configuration with mixed queue models
file_path = os.path.join(PATH, data_folder, cs_folder)
opti_cs_cfg = load(file_path, "optimal_qn_model.csv")
print("Queue Network Configuration:")
# print(opti_cs_cfg)
opti_cs_cfg.head()

In [ ]:
# extract parameters from the configuration DataFrame
# and casting them to proper types
nodes = list(opti_cs_cfg["node"].values.astype(int))
names = list(opti_cs_cfg["name"].values)
types = list(opti_cs_cfg["type"].values)
mius = list(opti_cs_cfg["miu"].values)
lambda_zs = list(opti_cs_cfg["lambda0"].values)
n_servers = list(opti_cs_cfg["s"].values.astype(int))
kaps = list(opti_cs_cfg["K"].values.astype(float))

# Convert K=0=nan to understandable infinite capacity -> None
for i in range(len(kaps)):
    if np.isnan(kaps[i]):
        kaps[i] = None
    else:
        kaps[i] = float(kaps[i])

# Convert string representations of arrays to actual numpy arrays
# and create routing matrix P
prob = []
for pm_str in opti_cs_cfg["pm"].values:
    pm_values = pm_str.strip("[]").split(",")
    pm_values = [float(val) for val in pm_values]
    prob.append(pm_values)
P = np.array((prob))

# create queue objects based on the configuration
queues = []
for i in range(len(mius)):
    # print(f"Creating queue: '{names[i]}' for Node: {nodes[i]}")
    # print(f"  Type: {types[i]}, λ: {lambda_zs[i]}, μ: {mius[i]}, s: {n_servers[i]}, K: {kaps[i]}")
    # create queue object
    q = Queue(
        model=types[i],
        _lambda=lambda_zs[i],
        miu=mius[i],
        n_servers=n_servers[i],
        kapacity=kaps[i],
    )
    # add queue to the list
    queues.append(q)

In [ ]:
# Solve the network analytically
# first node metrics
opti_analyt_nd_metrics = solve_jackson_network(mius, lambda_zs, queues, P)

# then network metrics
opti_analyt_net_metrics = calculate_net_metrics(opti_analyt_nd_metrics)

In [ ]:
print("\n--- Analytical Jackson Network (Node Metrics) ---")
# print(opti_analyt_nd_metrics)
opti_analyt_nd_metrics.head()

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_analytical_node_metrics.csv", opti_analyt_nd_metrics)

In [ ]:
print("\n--- Analytical Jackson Network (Network-wide Metrics) ---")
# print(opti_analyt_net_metrics)
opti_analyt_net_metrics.head()

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_analytical_net_metrics.csv", opti_analyt_net_metrics)

In [ ]:
# plotting the queue network with metrics on each node
# data table column names
col_names =[
    "Component",
    "$\mathbf{\\lambda}$ [req/s]",
    "$\mathbf{\\mu}$ [req/s]",
    "$\mathbf{\\rho}$",
    "$\mathbf{L}$ [req]",
    "$\mathbf{L_q}$ [req]",
    "$\mathbf{W}$ [s/req]",
    "$\mathbf{W_q}$ [s/req]"
]

node_names = opti_cs_cfg["name"].values.tolist()
print(f"Datatable column names: {col_names}")
print(f"Node names: {node_names}")  

# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# Plot the queue network
plot_queue_network(P,
                   opti_analyt_net_metrics,
                   opti_analyt_nd_metrics,
                   node_names,
                   col_names,
                   file_path,
                   "opti_analytical_qn_diagram.png")

#### **Numerical Solution (Simulation-Based)**

##### **Base Configuration**

##### **Optimized Configuration**

## **Results**

### **Saving Results**

## **Analysis**

### **Analytical Solution Graphs**

### **Numerical Solution Graphs**

## **Conclusion**

## **Future Work**

## **References & Sources**
<!-- TODO fix the references, links and details -->
1. [Queueing Theory](https://en.wikipedia.org/wiki/Queueing_theory)
2. [Dimensional Analysis](https://en.wikipedia.org/wiki/Dimensional_analysis)
3. [Simulation in Healthcare](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6466220/)

---

# **HASTA AKI!!!**

NOTES:

- $\Pi_{0}$ should be the x-axis.
- $\Pi_{1}$ should be the y-axis.
- $\Pi_{3}$ should be the contour.
- $\Pi_{2}$ should be the color of the lines.

Understanding Contour Behavior in Your Queue System Chart
The contour lines in your plot represent queue length values (10, 20, 30, 40, 50), and they appear higher and closer to the Y-axis for higher values due to several important queue theory principles:

Why This Pattern Occurs
Exponential Queue Growth: In queueing theory, as system utilization (ρ = λ/μ) approaches 1.0, queue length grows exponentially rather than linearly. This fundamental property creates the compressed contour pattern you're seeing.

Dimensionless Variables Relationship: Your dimensionless plot shows that:

Small changes in the X-axis variable (first Pi coefficient) cause large changes in queue length when the system is near capacity
The Y-axis variable (second Pi coefficient) has less impact on queue length at higher utilization levels
Log-Log Scale Effect: You're using logarithmic scales on both axes, which compresses the higher values and spreads out the lower values, making this pattern more pronounced.

Queue Theory Interpretation
This pattern visualizes a critical queueing system concept: the performance cliff effect. As your system approaches saturation:

Small increases in load (moving right on X-axis) cause dramatically larger queue lengths
Improving service rate (moving up on Y-axis) provides diminishing returns once the system is near saturation
This is why the higher contour lines (40, 50) are compressed toward the right side of the chart and appear to "stack up" near the Y-axis.

Engineering Significance
This pattern in your data suggests:

Your system has a critical utilization threshold beyond which performance degrades rapidly
The "High Utilization Region" you've marked represents the danger zone where small load increases cause large queue growth
The system should be operated with sufficient margin from this threshold to maintain stable performance
This is exactly the kind of relationship a Moody-style chart should reveal - helping identify safe operating regions and performance boundaries in your system.